# Taller Interactivo: Movimiento de Proyectiles y Aplicaciones en Ingeniería
### Física BR — Física Mecánica
**Profesor:** Juan David Betancur Ríos · **Semestre:** 2026-2

---
**Versión para Google Colab.** Cada punto es autónomo e incluye esquema del montaje,
sliders, botón **"Mostrar solución"** (respuesta oculta hasta que la pidas) y cuestionario.

> **Uso:** `Entorno de ejecución → Ejecutar todo`. Si un slider no aparece, re-ejecuta esa celda.

Gravedad:  g = 9,81 m/s²  (sin resistencia del aire)

---
> © 2026 **Juan David Betancur Ríos**, docente de física para ingeniería. Material educativo de uso académico (Física BR). Todos los derechos reservados.
> Material de uso académico. Todos los derechos reservados. Prohibida su reproducción o distribución sin autorización de los autores.
> *Elaborado con apoyo de herramientas de inteligencia artificial, bajo la supervisión y criterio pedagógico del autor.*

---
## Punto 1 — Lanzamiento de un dron de carga (Ing. Mecánica/Aeronáutica)
Proyectil con rapidez  v₀ = 20,0 m/s  y ángulo  θ = 45°  desde el suelo.

- vₓ = v₀·cos(θ),   v_y = v₀·sen(θ)
- Tiempo de vuelo:  t = 2·v₀·sen(θ)/g
- Alcance:  R = v₀²·sen(2θ)/g       Altura máxima:  H = v₀²·sen²(θ)/(2·g)

> 🔧 **Aplicación en ingeniería:** el lanzamiento de un dron de carga (entregas, agricultura, inspección de ductos) requiere prever su trayectoria para alcanzar el objetivo. El ángulo y la velocidad iniciales determinan el alcance útil.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))


g = 9.81
# --- fin setup ---


def esquema():
    fig,ax=plt.subplots(figsize=(5,2.5))
    th=np.radians(45); t=np.linspace(0,2*20*np.sin(th)/g,50)
    ax.plot(20*np.cos(th)*t, 20*np.sin(th)*t-0.5*g*t**2, color='navy')
    ax.annotate('',xy=(3,3),xytext=(0,0),arrowprops=dict(arrowstyle='->',color='red',lw=2))
    ax.text(1.2,2.2,'v₀',color='red'); ax.text(2.2,0.3,'θ')
    ax.axhline(0,color='k',lw=.5); ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title('Lanzamiento con ángulo desde el suelo'); ax.set_aspect('equal'); plt.tight_layout(); plt.show()
esquema()

def tiro(v0=20.0, ang=45.0):
    th=np.radians(ang); vx=v0*np.cos(th); vy=v0*np.sin(th)
    t_fl=2*vy/g; t=np.linspace(0,t_fl,200); x=vx*t; y=vy*t-0.5*g*t**2
    R=v0**2*np.sin(2*th)/g; H=vy**2/(2*g)
    plt.figure(figsize=(9,4.5)); plt.plot(x,y,lw=2,color='navy')
    plt.scatter([R],[0],color='red',s=60,zorder=5,label=f'alcance={R:.1f} m')
    plt.scatter([R/2],[H],color='green',s=60,zorder=5,label=f'H={H:.1f} m')
    plt.xlabel('x (m)'); plt.ylabel('y (m)'); plt.title(f'v₀={v0:.0f} m/s, θ={ang:.0f}°')
    plt.legend(); plt.grid(alpha=.3); plt.axhline(0,color='k',lw=.5); plt.tight_layout(); plt.show()

interact(tiro,
    v0=FloatSlider(value=20.0,min=5,max=50,step=1,description='v₀ (m/s)'),
    ang=FloatSlider(value=45.0,min=10,max=80,step=5,description='θ (grados)'));

def solucion():
    v0=20.0; th=np.radians(45)
    print(f"vₓ={v0*np.cos(th):.2f} m/s, v_y={v0*np.sin(th):.2f} m/s")
    print(f"Tiempo de vuelo = {2*v0*np.sin(th)/g:.3f} s")
    print(f"Alcance = {v0**2*np.sin(2*th)/g:.3f} m")
    print(f"Altura máxima = {(v0*np.sin(th))**2/(2*g):.3f} m")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 1 ---")
quiz_numerico("Tiempo de vuelo (v₀=20 m/s, θ=45°):",2.883,0.04,"s","t=2v₀senθ/g≈2,88 s.", id_preg="Proyecti_p1_1", peso=2.0)
quiz_numerico("Alcance R (v₀=20 m/s, θ=45°):",40.775,0.04,"m","R=v₀²sen(2θ)/g≈40,8 m.", id_preg="Proyecti_p1_2", peso=2.0)
quiz_opcion_multiple("¿Dónde la velocidad vertical v_y es cero?",
    ["Al inicio","En la altura máxima","Al caer","Nunca"],1,"En el punto más alto v_y=0.", id_preg="Proyecti_p1_3", peso=2.0)


---
## Punto 2 — Entrega desde un dron en vuelo (Ing. Civil/Logística)
Paquete soltado con velocidad **horizontal**  v₀ = 15,0 m/s  desde  h = 45,0 m.

- Tiempo de caída:  t = √(2·h/g)
- Alcance:  x = v₀·t       Velocidad de impacto:  v = √(v₀² + (g·t)²)

> 🔧 **Aplicación en ingeniería:** soltar carga desde un dron en vuelo (paquetes, insumos agrícolas) exige calcular dónde caerá, considerando la velocidad del dron. Es logística aérea aplicada, cada vez más usada en zonas de difícil acceso.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))


g = 9.81
# --- fin setup ---


def esquema():
    fig,ax=plt.subplots(figsize=(5,3))
    ax.plot([0,0],[0,4],'k-',lw=1); ax.text(-0.3,4,'dron',ha='right')
    t=np.linspace(0,1,50); ax.plot(3*t, 4-4*t**2, color='darkgreen')
    ax.annotate('',xy=(1.2,4),xytext=(0,4),arrowprops=dict(arrowstyle='->',color='red',lw=2))
    ax.text(0.6,4.2,'v₀',color='red'); ax.text(-0.3,2,'h',ha='right')
    ax.axhline(0,color='brown',lw=2); ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title('Lanzamiento horizontal desde altura'); plt.tight_layout(); plt.show()
esquema()

def caida(v0=15.0, h=45.0):
    t_fall=np.sqrt(2*h/g); t=np.linspace(0,t_fall,200); x=v0*t; y=h-0.5*g*t**2
    plt.figure(figsize=(9,4.5)); plt.plot(x,y,lw=2,color='darkgreen')
    plt.scatter([v0*t_fall],[0],color='red',s=60,zorder=5,label=f'impacto x={v0*t_fall:.1f} m')
    plt.xlabel('x (m)'); plt.ylabel('y (m)'); plt.title(f'v₀={v0:.0f} m/s desde h={h:.0f} m')
    plt.legend(); plt.grid(alpha=.3); plt.axhline(0,color='k',lw=.5); plt.tight_layout(); plt.show()

interact(caida,
    v0=FloatSlider(value=15.0,min=5,max=40,step=1,description='v₀ (m/s)'),
    h=FloatSlider(value=45.0,min=10,max=100,step=5,description='h (m)'));

def solucion():
    v0=15.0; h=45.0; t=np.sqrt(2*h/g)
    print(f"Tiempo de caída = √(2h/g) = {t:.3f} s")
    print(f"Alcance horizontal = v₀·t = {v0*t:.3f} m")
    print(f"Velocidad de impacto = {np.hypot(v0,g*t):.3f} m/s")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 2 ---")
quiz_numerico("Tiempo de caída (h=45 m):",3.029,0.04,"s","t=√(2h/g)≈3,03 s. No depende de v₀.", id_preg="Proyecti_p2_4", peso=2.0)
quiz_numerico("Alcance horizontal (v₀=15 m/s, h=45 m):",45.434,0.04,"m","x=v₀t≈45,4 m.", id_preg="Proyecti_p2_5", peso=2.0)
quiz_opcion_multiple("Con MAYOR velocidad horizontal, el tiempo de caída:",
    ["Aumenta","Disminuye","No cambia: la caída vertical es independiente de v₀","Se duplica"],2,
    "El tiempo de caída solo depende de h.", id_preg="Proyecti_p2_6", peso=2.0)


---
## Punto 3 — Ángulo óptimo de alcance (Ing. Industrial/Deportiva)
Con rapidez fija  v₀ = 20,0 m/s, se busca el ángulo de mayor alcance.

**Alcance:**  R = v₀²·sen(2θ)/g  → máximo cuando 2θ=90°, es decir θ=45°.

> 🔧 **Aplicación en ingeniería:** maximizar el alcance (45° sin resistencia) aparece en aspersión de cultivos, sistemas de riego y balística de proyectos. Conocer el ángulo óptimo ahorra energía y mejora la cobertura.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))


g = 9.81
# --- fin setup ---


def esquema():
    fig,ax=plt.subplots(figsize=(5,2.5))
    for a,c in [(30,'orange'),(45,'green'),(60,'purple')]:
        th=np.radians(a); t=np.linspace(0,2*20*np.sin(th)/g,50)
        ax.plot(20*np.cos(th)*t,20*np.sin(th)*t-0.5*g*t**2,color=c,label=f'{a}°')
    ax.axhline(0,color='k',lw=.5); ax.legend(); ax.set_title('Trayectorias a distintos ángulos')
    ax.set_xlabel('x'); ax.set_ylabel('y'); plt.tight_layout(); plt.show()
esquema()

def alcance_vs_angulo(v0=20.0, ang_marcado=45.0):
    angs=np.linspace(0,90,300); R=v0**2*np.sin(2*np.radians(angs))/g
    Rm=v0**2*np.sin(2*np.radians(ang_marcado))/g
    plt.figure(figsize=(9,4.5)); plt.plot(angs,R,lw=2,color='purple')
    plt.axvline(45,color='green',ls='--',label='óptimo 45°')
    plt.scatter([ang_marcado],[Rm],color='red',s=60,zorder=5,label=f'θ={ang_marcado:.0f}°→{Rm:.1f} m')
    plt.xlabel('ángulo θ (grados)'); plt.ylabel('alcance R (m)'); plt.title(f'Alcance vs ángulo (v₀={v0:.0f} m/s)')
    plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

interact(alcance_vs_angulo,
    v0=FloatSlider(value=20.0,min=5,max=50,step=1,description='v₀ (m/s)'),
    ang_marcado=FloatSlider(value=45.0,min=5,max=85,step=5,description='θ (grados)'));

def solucion():
    v0=20.0
    print(f"Alcance máximo (θ=45°) = v₀²/g = {v0**2/g:.3f} m")
    print("Ángulos complementarios (ej. 30° y 60°) dan el mismo alcance.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("Alcance máximo (v₀=20 m/s, θ=45°):",40.775,0.04,"m","R_max=v₀²/g≈40,8 m.", id_preg="Proyecti_p3_7", peso=2.0)
quiz_opcion_multiple("¿Qué ángulo da el alcance máximo (sin aire)?",["30°","45°","60°","90°"],1,
    "sen(2θ)=1 ⟹ θ=45°.", id_preg="Proyecti_p3_8", peso=2.0)
quiz_opcion_multiple("¿Qué ángulos dan el MISMO alcance?",
    ["Ninguno","Complementarios (ej. 30° y 60°)","Solo 45°","Iguales"],1,
    "θ y 90°−θ dan igual alcance.", id_preg="Proyecti_p3_9", peso=2.0)


---
## Punto 4 — Velocidad y posición en un instante (Ing. Mecánica/Control)
Proyectil con  v₀ = 25,0 m/s,  θ = 50°.  Se analiza el estado (posición y velocidad)
en un instante intermedio  t = 1,50 s  (útil para control y seguimiento de trayectorias).

- Posición:  x = vₓ·t,   y = v_y·t − ½·g·t²
- Velocidad:  vₓ constante,   v_y(t) = v₀·sen(θ) − g·t,   |v| = √(vₓ² + v_y²)

> 🔧 **Aplicación en ingeniería:** conocer la velocidad y posición en un instante permite programar la interceptación, el seguimiento o la entrega precisa en sistemas automatizados (drones, brazos robóticos, aspersores). Es control de movimiento en tiempo real.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))


g = 9.81
# --- fin setup ---


def esquema():
    fig,ax=plt.subplots(figsize=(5,2.8))
    v0=25; th=np.radians(50); t_fl=2*v0*np.sin(th)/g; t=np.linspace(0,t_fl,80)
    ax.plot(v0*np.cos(th)*t, v0*np.sin(th)*t-0.5*g*t**2, color='navy')
    ti=1.5; xi=v0*np.cos(th)*ti; yi=v0*np.sin(th)*ti-0.5*g*ti**2
    vxi=v0*np.cos(th); vyi=v0*np.sin(th)-g*ti
    ax.scatter([xi],[yi],color='red',s=50,zorder=5)
    ax.annotate('',xy=(xi+vxi*0.3,yi+vyi*0.3),xytext=(xi,yi),arrowprops=dict(arrowstyle='->',color='red',lw=2))
    ax.text(xi+1,yi+1,'v(t)',color='red'); ax.axhline(0,color='k',lw=.5)
    ax.set_title('Estado del proyectil en un instante t'); ax.set_xlabel('x'); ax.set_ylabel('y')
    plt.tight_layout(); plt.show()
esquema()

def instante(v0=25.0, ang=50.0, t_inst=1.5):
    th=np.radians(ang); vx=v0*np.cos(th); vy0=v0*np.sin(th)
    t_fl=2*vy0/g; t=np.linspace(0,t_fl,200); x=vx*t; y=vy0*t-0.5*g*t**2
    xi=vx*t_inst; yi=vy0*t_inst-0.5*g*t_inst**2; vyi=vy0-g*t_inst; vi=np.hypot(vx,vyi)
    plt.figure(figsize=(9,4.5)); plt.plot(x,y,lw=2,color='navy',alpha=.6)
    plt.scatter([xi],[yi],color='red',s=70,zorder=5,label=f't={t_inst:.1f}s: ({xi:.1f}, {yi:.1f}) m')
    plt.quiver(xi,yi,vx,vyi,color='red',scale=80,width=.006,label=f'|v|={vi:.1f} m/s')
    plt.xlabel('x (m)'); plt.ylabel('y (m)'); plt.title(f'v₀={v0:.0f} m/s, θ={ang:.0f}°, instante t={t_inst:.1f}s')
    plt.legend(); plt.grid(alpha=.3); plt.axhline(0,color='k',lw=.5); plt.tight_layout(); plt.show()

interact(instante,
    v0=FloatSlider(value=25.0,min=10,max=50,step=1,description='v₀ (m/s)'),
    ang=FloatSlider(value=50.0,min=10,max=80,step=5,description='θ (grados)'),
    t_inst=FloatSlider(value=1.5,min=0.2,max=4,step=0.1,description='t (s)'));

def solucion():
    v0=25.0; th=np.radians(50); ti=1.5
    vx=v0*np.cos(th); vy0=v0*np.sin(th); vyi=vy0-g*ti
    print(f"Posición: x={vx*ti:.2f} m, y={vy0*ti-0.5*g*ti**2:.2f} m")
    print(f"Velocidad: vₓ={vx:.2f} m/s (constante), v_y={vyi:.2f} m/s")
    print(f"Rapidez |v| = √(vₓ²+v_y²) = {np.hypot(vx,vyi):.2f} m/s")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 4 ---")
quiz_numerico("Altura y en t=1,5 s (v₀=25 m/s, θ=50°):",17.69,0.05,"m",
    "y=v₀senθ·t−½gt²≈17,7 m.", id_preg="Proyecti_p4_10", peso=2.0)
quiz_numerico("Rapidez |v| en t=1,5 s:",16.67,0.05,"m/s",
    "vₓ=16,07, v_y=4,44 ⟹ |v|≈16,7 m/s.", id_preg="Proyecti_p4_11", peso=2.0)
quiz_opcion_multiple("¿Qué componente de la velocidad permanece constante?",
    ["La vertical v_y","La horizontal vₓ (no hay fuerza horizontal)","Ninguna","Ambas"],1,
    "Sin aire, no hay aceleración horizontal, así que vₓ es constante.", id_preg="Proyecti_p4_12", peso=2.0)


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))

# Calificación formativa del taller
calificacion_final(total_preguntas=12)


---
## ✅ Fin del taller de Proyectiles (4 puntos)
Experimenta con los sliders y **predice** antes de pulsar "Mostrar solución".
*Física Mecánica — 2026-2.*